# Cleaned PCB v1i YOLOv8 Detection Baseline in Google Colab

This notebook is for the **cleaned PCB-Defect-Detection.v1i.coco detection baseline only**.

- Model: `yolov8n.pt`
- Task: object detection
- Training style: fine-tuning / transfer learning from pretrained weights
- Dataset config: `configs/pcb_v1i_coco_baseline.yaml`
- Scope: cleaned dataset only, no PKU, no DeepPCB, no tiling, no segmentation, no severity yet


## Colab Setup Notes

Use one of these project access options:

1. Put the whole repo in Google Drive and mount Drive in Colab.
2. Clone your GitHub repo into `/content/`.
3. Upload the project folder manually if needed.

The notebook below assumes **Google Drive** by default because it is the safest way to keep datasets, runs, and weights between sessions.


In [ ]:
!pip install -q ultralytics==8.4.14 opencv-python pyyaml matplotlib pandas

import platform
import sys
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    !nvidia-smi
else:
    print('GPU not detected. In Colab, switch Runtime > Change runtime type > GPU before training.')


## Mount Google Drive and Open the Repo

Update `PROJECT_ROOT` if your repo folder has a different Drive location.

If you prefer to clone the repo instead of using Drive, skip the mount lines and set `PROJECT_ROOT` to the cloned folder path.


In [ ]:
from pathlib import Path

USE_DRIVE = True
PROJECT_ROOT = Path('/content/drive/MyDrive/PCB-Defect-Detector')  # change if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

assert PROJECT_ROOT.exists(), f'Update PROJECT_ROOT to your repo folder: {PROJECT_ROOT}'

%cd $PROJECT_ROOT
print('Working directory:', PROJECT_ROOT)


## Check Dataset Paths and Prepare the YOLO Workspace

This reuses the repo's cleaned-dataset training script and config.
It validates the `train` / `valid` / `test` split paths and generates the Ultralytics data YAML without starting training yet.


In [ ]:
from pathlib import Path

CONFIG_PATH = Path('configs/pcb_v1i_coco_baseline.yaml')
SCRIPT_PATH = Path('scripts/pcb_v1i_yolov8_colab_train.py')
DATA_YAML_PATH = Path('configs/pcb_v1i_yolov8_baseline_data.yaml')

assert CONFIG_PATH.exists(), f'Missing config: {CONFIG_PATH}'
assert SCRIPT_PATH.exists(), f'Missing training script: {SCRIPT_PATH}'

!python scripts/pcb_v1i_yolov8_colab_train.py --prepare-only
!cat configs/pcb_v1i_yolov8_baseline_data.yaml


## Baseline Training Settings

This keeps the baseline simple and detection-only.
Split usage stays fixed as:

- `train` for training
- `valid` for validation
- `test` for the post-training inference check


In [ ]:
MODEL = 'yolov8n.pt'
EPOCHS = 10
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
FRACTION = 1.0
PATIENCE = 20
PROJECT_DIR = 'runs/pcb_v1i_baseline'
RUN_NAME = 'yolov8n_pcb_v1i_baseline_colab'

print({
    'model': MODEL,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'device': DEVICE,
    'fraction': FRACTION,
    'patience': PATIENCE,
    'project_dir': PROJECT_DIR,
    'run_name': RUN_NAME,
})


In [ ]:
import subprocess

train_cmd = [
    'python', 'scripts/pcb_v1i_yolov8_colab_train.py',
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--device', DEVICE,
    '--fraction', str(FRACTION),
    '--patience', str(PATIENCE),
    '--project', PROJECT_DIR,
    '--name', RUN_NAME,
]

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


## Check the Training Outputs

Ultralytics may place the run under `runs/detect/...`, so this cell checks both likely output locations.


In [ ]:
from pathlib import Path
import pandas as pd

candidate_run_dirs = [
    Path(PROJECT_DIR) / RUN_NAME,
    Path('runs/detect') / PROJECT_DIR / RUN_NAME,
]

RUN_DIR = None
for candidate in candidate_run_dirs:
    if candidate.exists():
        RUN_DIR = candidate
        break

assert RUN_DIR is not None, 'Training run folder not found yet.'

print('Run directory:', RUN_DIR)
print('Best weights:', RUN_DIR / 'weights' / 'best.pt')
print('Last weights:', RUN_DIR / 'weights' / 'last.pt')

results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    display(pd.read_csv(results_csv).tail())


## Quick Validation / Inference Check

The training script already saves a post-training prediction check on the `test` split.
This cell displays those saved outputs if they exist, or regenerates them from `best.pt` if needed.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

best_weights = RUN_DIR / 'weights' / 'best.pt'
assert best_weights.exists(), f'Missing best weights: {best_weights}'

test_dir = Path('data/resources/PCB-Defect-Detection.v1i.coco/test')
sample_images = sorted(test_dir.glob('*.jpg'))[:8]
assert sample_images, f'No test images found in {test_dir}'

pred_root = Path('data/inspection_outputs')
pred_name = f'{RUN_NAME}_predictions'
pred_dir = pred_root / pred_name

if not pred_dir.exists() or not any(pred_dir.glob('*.jpg')):
    model = YOLO(str(best_weights))
    results = model.predict(
        source=[str(path) for path in sample_images],
        imgsz=IMGSZ,
        conf=0.25,
        device=DEVICE,
        save=True,
        project=str(pred_root),
        name=pred_name,
        exist_ok=True,
        verbose=False,
    )
    print('Predictions generated for', len(results), 'images')

print('Prediction directory:', pred_dir)
for image_path in sorted(pred_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(image_path)))


## Notes

- This notebook is for the cleaned PCB v1i dataset only.
- It keeps the workflow detection-only with pretrained `yolov8n.pt`.
- PKU, DeepPCB, tiling, segmentation, severity, augmentation, and dataset merging stay for later steps.
